In [ ]:
from utils import find_chunk_boundaries, TOKENIZE_PATTERN, get_pre_tokens_count

from multiprocessing import Pool
import regex as re

PARALLELIZE = True

SPECIAL_TOKENS = ["<|endoftext|>"]  # Add more special tokens as needed

filename = "./data/TinyStoriesV2-GPT4-valid.txt"
filename = "../../data/test_data.txt"

In [ ]:
enable_mp = True
input_path = filename
num_process = 4

In [ ]:
with open(input_path, 'rb') as f:
            boundaries = find_chunk_boundaries(f, num_process, b"<|endoftext|>")

            # only for testing purposes, limit to the first 4 chunks
            # TODO: remove this line after testing
            boundaries = boundaries[:2]

if enable_mp:
    pairs = list(zip(boundaries[:-1], boundaries[1:]))
    args = [(input_path, start, end, SPECIAL_TOKENS) for start, end in pairs]

    with Pool(processes=num_process) as pool:
            #results = pool.starmap(pre_tokenize_chunk, args)
            chunks_count = pool.starmap(get_pre_tokens_count, args)

    # combine results in a single dictionary
    pre_tokens_count = {}
    for chunk in chunks_count:
        for term, count in chunk.items():
            if term in pre_tokens_count:
                pre_tokens_count[term] += count
            else:
                pre_tokens_count[term] = count


In [ ]:
pre_tokens_count

In [ ]:
del pre_tokens_count['\n']

In [ ]:
pre_tokens_count

In [ ]:
def _convert_key_to_tuple_of_bytes(key):
        """
        Convert a key to bytes.
        """
        return tuple(c.encode('utf-8') for c in key)


In [ ]:
def _convert_key_to_tuple_of_bytes_v2(key):
        """
        Convert a key to bytes.
        """
        return tuple(bytes([b]) for b in key.encode('utf-8'))

In [ ]:
pre_tokens_count_bytes = {_convert_key_to_tuple_of_bytes(k): v for k, v in pre_tokens_count.items()}


In [ ]:
pre_tokens_count_bytes

In [ ]:
def _get_pair_freq(freq_dict: dict):
        """ Calculate the frequency of each pair of consecutive bytes in the input dictionary.
        Args:
            freq_dict (dict): A dictionary where keys are tuples of bytes and values are their frequencies.

        Returns:
            dict: A dictionary with pairs of consecutive bytes as keys and their frequencies as values.
        """
        pairs_freq_dict = {}
        for key, value in freq_dict.items():
            for first, second in zip(key, key[1:]):
                pair = (first, second)
                if pair in pairs_freq_dict:
                    pairs_freq_dict[pair] += value
                else:
                    pairs_freq_dict[pair] = value

               
        return pairs_freq_dict

In [ ]:
from collections import defaultdict

In [ ]:
def _get_pair_dict_and_freq(freq_dict: dict):
        """ Calculate the frequency of each pair of consecutive bytes in the input dictionary.
        Args:
            freq_dict (dict): A dictionary where keys are tuples of bytes and values are their frequencies.

        Returns:
            dict: A dictionary with pairs of consecutive bytes as keys and their frequencies as values.
        """
        pair_pre_tokens_dict = defaultdict(set)
        pairs_freq_dict = {}
        for key, value in freq_dict.items():
            for first, second in zip(key, key[1:]):
                pair = (first, second)
                if pair in pairs_freq_dict:
                    pairs_freq_dict[pair] += value
                else:
                    pairs_freq_dict[pair] = value

                pair_pre_tokens_dict[pair].add(key)
        return pairs_freq_dict, pair_pre_tokens_dict

In [ ]:
pre_tokens_count_bytes

In [ ]:
pair_freq_dict = _get_pair_freq(pre_tokens_count_bytes)
pair_freq_dict, pair_pre_tokens_dict = _get_pair_dict_and_freq(pre_tokens_count_bytes)

In [ ]:
pair_pre_tokens_dict

In [ ]:
pair_freq_dict

In [ ]:
def _get_top_pair(stats):
    return max(stats, key=lambda p: (stats[p], p))

In [ ]:
top_pair = _get_top_pair(pair_freq_dict)

In [ ]:
top_pair

ora che ho trovato la top pair, devo fare il merge sostituendo nei pre tokens la coppia 's', 't' con 'st'

In [ ]:
for pre_tokens, count in pre_tokens_count_bytes.items():
    # Your code to process each pair goes here
    break

In [ ]:
pre_tokens

cerco solo i pre tokens che sono interessati da questa sostituzione

In [ ]:
top_pair

In [ ]:
pre_tokens_to_change = pair_pre_tokens_dict.get(top_pair)
pre_tokens_to_change

# funzione per modificare il pre token inserendo la top pair

In [ ]:
def _merge_pair_in_token(pair, token):
    first, second = pair
    merged_token = []
    i = 0
    while i < len(token):
        if i < len(token) - 1 and token[i] == first and token[i + 1] == second:
            merged_token.append(first + second)
            i += 2
        else:
            merged_token.append(token[i])
            i += 1
    return tuple(merged_token)

In [ ]:
test_tuple = (b'a', b's', b't', b'a', b'r', b's', b't') 
test_top_pair = (b's', b't')
assert _merge_pair_in_token(test_top_pair, test_tuple) == (b'a', b'st', b'a', b'r', b'st')
# Expected output: (b'a', b'st', b'a', b'r', b'st')

In [ ]:
test_top_pair = (b'a', b's')
assert _merge_pair_in_token(test_top_pair, test_tuple) == (b'as', b't', b'a', b'r', b's', b't')
# Expected output: (b'ast', b'a', b'r', b'st')


applico la sostituzione

In [ ]:
pre_tokens_count_bytes

In [ ]:
pre_tokens_to_change

# creo il nuovo dict per il conteggio dei pre tokens

In [ ]:
pre_tokens_to_change

In [ ]:
top_pair

In [ ]:
??_get_pair_freq

In [ ]:
def _get_pair_from_token(token):
    pair_list = []
    for i in range(len(token) - 1):
        pair_list.append((token[i], token[i + 1]))
    return pair_list

In [ ]:
update_pair_freq_dict = pair_freq_dict.copy()

In [ ]:
sorted(pair_pre_tokens_dict.keys())

In [ ]:
new_pre_tokens_count_bytes = pre_tokens_count_bytes.copy()
new_pair_pre_tokens_dict = pair_pre_tokens_dict.copy()

for pre_tokens in pre_tokens_to_change:
    new_pre_tokens = _merge_pair_in_token(top_pair, pre_tokens)
    old_count = pre_tokens_count_bytes.get(pre_tokens)
    if new_pre_tokens in new_pre_tokens_count_bytes:
        new_pre_tokens_count_bytes[new_pre_tokens] += old_count
    else:
        new_pre_tokens_count_bytes[new_pre_tokens] = old_count

    # get the pair from pre_tokens
    old_pair_list = _get_pair_from_token(pre_tokens)
    for pair in old_pair_list:
        # decrement the frequency of the old pair by the old count
        update_pair_freq_dict[pair] = update_pair_freq_dict.get(pair, 0) - old_count

        # remove old pre tokens
        pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
        if pre_tokens_list:
            pre_tokens_list.remove(pre_tokens)

        new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)

    new_pair_list = _get_pair_from_token(new_pre_tokens)
    for pair in new_pair_list:
        update_pair_freq_dict[pair] = update_pair_freq_dict.get(pair, 0) + old_count

        # add new pre tokens
        pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
        if pre_tokens_list:
            pre_tokens_list.append(new_pre_tokens)
        else:
            pre_tokens_list = [new_pre_tokens]

        new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)
    del new_pre_tokens_count_bytes[pre_tokens]


# remove pair with zero frequency
pairs_to_remove = [pair for pair, freq in update_pair_freq_dict.items() if freq == 0]
for pair in pairs_to_remove:
    del update_pair_freq_dict[pair]



In [ ]:
pre_tokens_count_bytes

In [ ]:
import copy

In [ ]:
new_pre_tokens_count_bytes = pre_tokens_count_bytes.copy()
new_pair_pre_tokens_dict = copy.deepcopy(pair_pre_tokens_dict)

for pre_tokens in pre_tokens_to_change:
    new_pre_tokens = _merge_pair_in_token(top_pair, pre_tokens)
    old_count = pre_tokens_count_bytes.get(pre_tokens)
    if new_pre_tokens in new_pre_tokens_count_bytes:
        new_pre_tokens_count_bytes[new_pre_tokens] += old_count
    else:
        new_pre_tokens_count_bytes[new_pre_tokens] = old_count

    # get the pair from pre_tokens
    old_pair_list = _get_pair_from_token(pre_tokens)
    for pair in old_pair_list:
        # decrement the frequency of the old pair by the old count
        update_pair_freq_dict[pair] = update_pair_freq_dict.get(pair, 0) - old_count

        # remove old pre tokens
        pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
        if pre_tokens_list:
            pre_tokens_list.remove(pre_tokens)

        new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)

    new_pair_list = _get_pair_from_token(new_pre_tokens)
    for pair in new_pair_list:
        update_pair_freq_dict[pair] = update_pair_freq_dict.get(pair, 0) + old_count

        # add new pre tokens
        pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
        if pre_tokens_list:
            pre_tokens_list.append(new_pre_tokens)
        else:
            pre_tokens_list = [new_pre_tokens]

        new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)
    del new_pre_tokens_count_bytes[pre_tokens]


# remove pair with zero frequency
pairs_to_remove = [pair for pair, freq in update_pair_freq_dict.items() if freq == 0]
for pair in pairs_to_remove:
    del update_pair_freq_dict[pair]



In [ ]:
pair

In [ ]:
new_pair_pre_tokens_dict

In [ ]:
pair_pre_tokens_dict.get(top_pair)

In [ ]:
temp = pair_pre_tokens_dict[(b'i', b'd')]

In [ ]:
temp.remove((b' ', b'w', b'i', b'd', b'e', b's', b't'))

In [ ]:
temp

In [ ]:
list(pre_tokens_to_change)[0]

In [ ]:
old_pair_list = _get_pair_from_token(list(pre_tokens_to_change)[0])

In [ ]:
old_pair_list

In [ ]:
pair_pre_tokens_dict[(b'i', b'd')]

In [ ]:
pair_pre_tokens_dict

In [ ]:
# make a test, compute update pair frequency dictionary iterating over new_pre_tokens_count_bytes
update_pair_freq_dict_test = _get_pair_freq(new_pre_tokens_count_bytes)

In [ ]:
assert update_pair_freq_dict == update_pair_freq_dict_test

# Create some functions

In [ ]:
def update_frequency_dict(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change):
    new_pre_tokens_count_bytes = pre_tokens_count_bytes.copy()
    new_pair_freq_dict = pair_freq_dict.copy()
    new_pair_pre_tokens_dict = pair_pre_tokens_dict.copy()

    for pre_tokens in pre_tokens_to_change:
        new_pre_tokens = _merge_pair_in_token(top_pair, pre_tokens)
        old_count = pre_tokens_count_bytes.get(pre_tokens)
        if new_pre_tokens in new_pre_tokens_count_bytes:
            new_pre_tokens_count_bytes[new_pre_tokens] += old_count
        else:
            new_pre_tokens_count_bytes[new_pre_tokens] = old_count

        # get the pair from pre_tokens
        old_pair_list = _get_pair_from_token(pre_tokens)
        for pair in old_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) - old_count
            # remove old pre tokens from the pair_pre_tokens_dict
            pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
            if pre_tokens_list:
                pre_tokens_list.remove(pre_tokens)
            new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)

        new_pair_list = _get_pair_from_token(new_pre_tokens)
        for pair in new_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) + old_count
            # add new pre tokens to the pair_pre_tokens_dict
            if pair not in new_pair_pre_tokens_dict:
                new_pair_pre_tokens_dict[pair] = set()
            pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
            if pre_tokens_list:
                pre_tokens_list.append(new_pre_tokens)
            else:
                pre_tokens_list = [new_pre_tokens]
            new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)
        del new_pre_tokens_count_bytes[pre_tokens]

    return new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict

In [ ]:
def update_frequency_dict_v2(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change):
    new_pre_tokens_count_bytes = pre_tokens_count_bytes.copy()
    new_pair_freq_dict = pair_freq_dict.copy()
    new_pair_pre_tokens_dict = pair_pre_tokens_dict.copy()

    for pre_tokens in pre_tokens_to_change:
        new_pre_tokens = _merge_pair_in_token(top_pair, pre_tokens)
        old_count = pre_tokens_count_bytes.get(pre_tokens)
        if new_pre_tokens in new_pre_tokens_count_bytes:
            new_pre_tokens_count_bytes[new_pre_tokens] += old_count
        else:
            new_pre_tokens_count_bytes[new_pre_tokens] = old_count

        # get the pair from pre_tokens
        old_pair_list = _get_pair_from_token(pre_tokens)
        for pair in old_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) - old_count

            if new_pair_freq_dict[pair] <= 0:
                del new_pair_freq_dict[pair]
                del new_pair_pre_tokens_dict[pair]
            else:
                # remove old pre tokens from the pair_pre_tokens_dict
                pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
                if pre_tokens_list:
                    pre_tokens_list.remove(pre_tokens)
                new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)

        new_pair_list = _get_pair_from_token(new_pre_tokens)
        for pair in new_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) + old_count
            # add new pre tokens to the pair_pre_tokens_dict
            if pair not in new_pair_pre_tokens_dict:
                new_pair_pre_tokens_dict[pair] = set()
            pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
            if pre_tokens_list:
                pre_tokens_list.append(new_pre_tokens)
            else:
                pre_tokens_list = [new_pre_tokens]
            new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)
        del new_pre_tokens_count_bytes[pre_tokens]

    return new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict

In [ ]:
new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict = update_frequency_dict(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change)

In [ ]:
new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict = update_frequency_dict_v2(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change)

In [ ]:
new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict

In [ ]:
_convert_key_to_tuple_of_bytes

In [ ]:
pre_tokens_count

In [ ]:
pre_tokens_count

In [ ]:
b'\xe2\x80\xa6'.decode('utf-8')

In [ ]:
'…'.encode('utf-8')

In [ ]:
pre_tokens_count['…'] = 5

In [ ]:
pre_tokens_count_bytes = {_convert_key_to_tuple_of_bytes(k): v for k, v in pre_tokens_count.items()}
pre_tokens_count_bytes

In [ ]:
pre_tokens_count_bytes = {_convert_key_to_tuple_of_bytes_v2(k): v for k, v in pre_tokens_count.items()}
pre_tokens_count_bytes

In [ ]:
tuple(c.encode('utf-8') for c in '…')

In [ ]:
list('…')

In [ ]:
list('abc')

In [ ]:
key='…'

In [ ]:
tuple(bytes([b]) for b in key.encode('utf-8'))   # encode whole string, then per-byte